# SMVT Top-3 Hit MD Simulation — GPU-Accelerated
**Target**: SMVT (SLC5A6) Na⁺-dependent multivitamin transporter
**Force field**: AMBER ff14SB + OpenFF Sage 2.1 + TIP3P
**Compounds**: Naftazone (−8.34), Phenobarbital (−8.30), Esketamine (−7.58)
**Protocol**: Minim → NVT 50→300K → NPT 100ps → Production 50ns

---
### Step 1 — Install & Detect GPU

In [ ]:
# ═══ Step 1: Install dependencies ═══
# Colab needs conda for OpenMM (system-level deps)

# 1a — Install condacolab (enables conda in Colab)
!pip install -q condacolab
import condacolab
condacolab.install()

# 1b — Install OpenMM via conda (handles C++ libs correctly)
!conda install -c conda-forge openmm openmmforcefields openff-toolkit pdbfixer -y 2>&1 | tail -5

# 1c — Install remaining via pip
!pip install -q rdkit-pypi validators tinydb lxml MDAnalysis matplotlib

# 1d — Detect GPU platform
import openmm as mm
print('OpenMM', mm.__version__)

PLATFORM_NAME = 'CPU'
PROPS = {}
for i in range(mm.Platform.getNumPlatforms()):
    name = mm.Platform.getPlatform(i).getName()
    if name == 'CUDA':
        PLATFORM_NAME = 'CUDA'
        PROPS = {'Precision': 'mixed'}
        break
    elif name == 'OpenCL' and PLATFORM_NAME == 'CPU':
        PLATFORM_NAME = 'OpenCL'
        PROPS = {'Precision': 'mixed'}

PROD_NS = 50 if PLATFORM_NAME == 'CUDA' else 5
print(f'Platform: {PLATFORM_NAME}  |  Production: {PROD_NS}ns')
if PLATFORM_NAME == 'CPU':
    print('WARNING: No GPU!  Runtime → Change runtime type → T4 GPU → Save → re-run')

# Verify imports
from openmmforcefields.generators import SystemGenerator
from openff.toolkit import Molecule as OFFMolecule
from openff.units import unit as off_unit
from pdbfixer import PDBFixer
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np, time, os
print('All imports OK — ready!')

### Step 2 — Mount Drive & Copy Files
Upload to `SMVT_MD/` on your Drive: `AF-Q9Y289-F1.pdb`, `NAFTAZONE_docked.pdbqt`, `PHENOBARBITAL_docked.pdbqt`, `ESKETAMINE_docked.pdbqt`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/SMVT_MD'
os.makedirs(DATA_DIR, exist_ok=True)
!cp -v /content/drive/MyDrive/SMVT_MD/* {DATA_DIR}/ 2>/dev/null
print('\nFiles in SMVT_MD/:')
!ls -la {DATA_DIR}/

### Step 3 — MD Pipeline (Run This for All 3 Compounds)

In [ ]:
RECEPTOR_PDB = f'{DATA_DIR}/AF-Q9Y289-F1.pdb'
OUT_DIR = f'{DATA_DIR}/trajectories'
os.makedirs(OUT_DIR, exist_ok=True)

TOP3 = [
    ('NAFTAZONE',         'C1=CC=C2C(=C1)C=CC(=O)C2=NNC(=N)N'),
    ('PHENOBARBITAL',     'CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2'),
    ('ESKETAMINE',        'CNC1(C2=CC=CC=C2Cl)CCCCC1=O'),
]

TEMP     = 300 * mm.unit.kelvin
PRESSURE = 1.0 * mm.unit.atmosphere
DT       = 2.0 * mm.unit.femtoseconds

def extract_pose(path):
    pos, in_m = [], False
    for line in open(path):
        if line.startswith('MODEL'):
            in_m = int(line.split()[1]) == 1
            if in_m: pos = []
        elif line.startswith('ENDMDL') and in_m: break
        elif in_m and (line[:4] == 'ATOM' or line[:6] == 'HETATM'):
            try: pos.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            except: pass
    return pos

def prep_protein(in_pdb, out_pdb):
    if os.path.exists(out_pdb): return out_pdb
    fixer = PDBFixer(filename=in_pdb)
    fixer.findMissingResidues(); fixer.findMissingAtoms()
    fixer.addMissingAtoms(); fixer.addMissingHydrogens(pH=7.4)
    with open(out_pdb,'w') as f: mm.app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
    return out_pdb

def run_one(name, smiles):
    odir = f'{OUT_DIR}/{name}'; os.makedirs(odir, exist_ok=True)
    chk  = f'{odir}/{name}_{PROD_NS}ns.chk'
    if os.path.exists(chk):
        print(f'[{name}] Already done, skipping')
        return

    print(f'\n{"="*50}\n[{name}] {PROD_NS}ns on {PLATFORM_NAME}\n{"="*50}')
    t0 = time.time()

    # 1—Protein
    prot = prep_protein(RECEPTOR_PDB, f'{odir}/protein.pdb')

    # 2—Vina pose
    vpos = extract_pose(f'{DATA_DIR}/{name}_docked.pdbqt')
    print(f'  Ligand atoms: {len(vpos)}')

    # 3—OpenFF molecule
    off = OFFMolecule.from_smiles(smiles, allow_undefined_stereo=True)
    off.generate_conformers(n_conformers=1)
    c = off.conformers[0]
    for i in range(min(off.n_atoms, len(vpos))):
        c[i] = np.array(vpos[i]) * off_unit.angstrom
    off.assign_partial_charges(partial_charge_method='gasteiger')

    # 4—RDKit PDB for Modeller
    rm = Chem.MolFromSmiles(smiles); rm = Chem.AddHs(rm)
    AllChem.EmbedMolecule(rm, randomSeed=42)
    rc = rm.GetConformer()
    for i in range(min(rm.GetNumAtoms(), len(vpos))):
        rc.SetAtomPosition(i, (vpos[i][0]/10., vpos[i][1]/10., vpos[i][2]/10.))
    lig_pdb = f'{odir}/ligand.pdb'
    Chem.MolToPDBFile(rm, lig_pdb)

    # 5—System generator
    print('  Parameterizing (OpenFF Sage)...')
    sg = SystemGenerator(
        forcefields=['amber/ff14SB.xml', 'amber/tip3p_standard.xml'],
        small_molecule_forcefield='openff-2.1.0',
        molecules=[off], cache=f'{odir}/cache.json')

    # 6—Solvate
    print('  Building solvated system...')
    pdb  = mm.app.PDBFile(prot)
    lig  = mm.app.PDBFile(lig_pdb)
    mod  = mm.app.Modeller(pdb.topology, pdb.positions)
    mod.add(lig.topology, lig.positions)
    mod.addSolvent(sg.forcefield, model='tip3p',
                   padding=0.8*mm.unit.nanometers,
                   ionicStrength=0.15*mm.unit.molar, neutralize=True)
    sys = sg.create_system(mod.topology, molecules=[off])
    print(f'  Particles: {sys.getNumParticles()}')

    # 7—Integrator + context
    integ = mm.LangevinMiddleIntegrator(TEMP, 1.0/mm.unit.picosecond, DT)
    plat  = mm.Platform.getPlatformByName(PLATFORM_NAME)
    sim   = mm.app.Simulation(mod.topology, sys, integ, plat, PROPS)
    sim.context.setPositions(mod.positions)

    # 8—Minimize
    print('  Minimizing (5000 steps)...')
    sim.minimizeEnergy(maxIterations=5000)

    # 9—NVT heating 50→300K
    print('  NVT heating 50→300K...')
    sim.integrator.setStepSize(1.0*mm.unit.femtoseconds)
    for t in [50, 150, 300]:
        sim.integrator.setTemperature(t*mm.unit.kelvin)
        sim.step(int(10*1000/1.0))
    sim.integrator.setTemperature(TEMP)
    sim.integrator.setStepSize(DT)
    sim.step(int(70*1000/2.0))

    # 10—NPT
    print('  NPT equil (100ps)...')
    sys.addForce(mm.MonteCarloBarostat(PRESSURE, TEMP))
    sim.context.reinitialize(preserveState=True)
    sim.step(int(100*1000/2.0))

    # 11—Production
    nstep   = int(PROD_NS * 1_000_000 / 2.0)
    sFreq   = int(100 * 1000 / 2.0)  # save every 100ps
    print(f'  Production: {PROD_NS}ns ({nstep} steps)...')
    sim.reporters.append(mm.app.DCDReporter(f'{odir}/{name}_{PROD_NS}ns.dcd', sFreq))
    sim.reporters.append(mm.app.StateDataReporter(
        f'{odir}/{name}_{PROD_NS}ns.csv', sFreq,
        step=True, time=True, potentialEnergy=True, kineticEnergy=True,
        temperature=True, volume=True, density=True))
    sim.reporters.append(mm.app.CheckpointReporter(chk, sFreq*10))
    sim.step(nstep)

    # 12—Save final frame
    state = sim.context.getState(getPositions=True)
    with open(f'{odir}/{name}_final.pdb','w') as f:
        mm.app.PDBFile.writeFile(sim.topology, state.getPositions(), f)
    sim.saveCheckpoint(chk)

    dt_min = (time.time()-t0)/60
    print(f'  [{name}] DONE  {dt_min:.1f} min  ({PROD_NS/dt_min:.1f} ns/min)')

    # Copy back to Drive
    !mkdir -p /content/drive/MyDrive/SMVT_MD/trajectories
    !cp -r {odir} /content/drive/MyDrive/SMVT_MD/trajectories/

# —— Run ——
for name, smi in TOP3:
    run_one(name, smi)

print('\nALL DONE. Trajectories saved to Drive → SMVT_MD/trajectories/')

### Step 4 — Quick RMSD Analysis (run after MD completes)

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

for name, _ in TOP3:
    dcd = f'{OUT_DIR}/{name}/{name}_{PROD_NS}ns.dcd'
    pdb = f'{OUT_DIR}/{name}/{name}_final.pdb'
    if not os.path.exists(dcd):
        print(f'{name}: trajectory not found'); continue
    
    u = mda.Universe(pdb, dcd)
    bb = u.select_atoms('protein and backbone')
    R = rms.RMSD(u, bb, ref_frame=0).run()
    
    rmsd = R.results.rmsd[:, 2]  # column 2 = RMSD after fitting
    t_ns = R.results.time / 1000
    ax.plot(t_ns, rmsd, label=f'{name} (mean {rmsd.mean():.2f} Å)')
    print(f'{name}: RMSD = {rmsd.mean():.2f} ± {rmsd.std():.2f} Å')

ax.set_xlabel('Time (ns)'); ax.set_ylabel('Backbone RMSD (Å)')
ax.set_title('SMVT Binding Stability — Top 3 Hits')
ax.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/rmsd_comparison.png', dpi=150)
plt.show()